# 01 — `pytest`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- écrire et lancer des tests avec `pytest`
- utiliser les fixtures (scope function/module/session)
- paramétrer un test avec `@pytest.mark.parametrize`
- utiliser `monkeypatch`, `tmp_path`, `caplog`
- organiser les tests avec des marks

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- classes, dataclasses, type hints
- modules, fichiers
- exceptions `try`/`except`

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- mocking (`unittest.mock`) — notebook 02
- factories (`factory_boy`) — notebook 03

## Plan

1. Pourquoi tester
2. Premier test : `assert`
3. Lancer `pytest`
4. Fixtures
5. `parametrize`
6. `monkeypatch`
7. `tmp_path` pour les fichiers
8. `caplog` pour les logs
9. Marks (`skip`, `xfail`, custom)
10. Synthèse
11. Exercices

---

## 1. Pourquoi tester

Un code non testé est un code qui casse **en silence**. Les tests :

- documentent le comportement attendu ;
- permettent de refactorer sans peur ;
- attrapent les régressions **avant** la mise en prod.

`pytest` est le framework de test standard de l'écosystème Python.

---

## 2. Premier test : `assert`

Un test `pytest` est une simple fonction dont le nom commence par `test_`.

In [ ]:
# Fichier : test_calcul.py

def double(n: int) -> int:
    return n * 2


def test_double_positif():
    assert double(3) == 6


def test_double_zero():
    assert double(0) == 0


def test_double_negatif():
    assert double(-2) == -4


Pour lancer : `uv run pytest test_calcul.py -v`

---

## 3. Lancer `pytest`

Commandes courantes :

```bash
pytest                     # tous les tests
pytest tests/              # dossier
pytest -v                  # verbose
pytest -x                  # stop au premier échec
pytest -k 'double'         # filtrer par nom
pytest --tb=short          # traceback court
```

---

## 4. Fixtures

Une fixture fournit un **contexte** réutilisable aux tests. Le paramètre du test qui porte le nom de la fixture est automatiquement injecté.

In [ ]:
import pytest


@pytest.fixture
def salle() -> dict:
    return {'nom': 'Mars', 'capacite': 12}


def test_nom(salle):
    assert salle['nom'] == 'Mars'


def test_capacite(salle):
    assert salle['capacite'] > 0


### Scope des fixtures

| Scope | Durée de vie |
|---|---|
| `function` (défaut) | Recréée pour chaque test |
| `module` | Recréée une fois par fichier |
| `session` | Recréée une seule fois pour toute la suite |

Usage : `@pytest.fixture(scope='module')`. Les fixtures lourdes (connexion DB) gagnent à être `session` pour éviter les recréations.

---

## 5. `parametrize`

Écrire un seul test, exécuté avec plusieurs jeux de données.

In [ ]:
@pytest.mark.parametrize('entree, attendu', [
    (0, 0),
    (1, 2),
    (5, 10),
    (-3, -6),
])
def test_double_parametrise(entree, attendu):
    assert double(entree) == attendu


---

## 6. `monkeypatch`

Fixture builtin pour modifier l'environnement pendant un test (variables d'env, stdin, attributs).

In [ ]:
import os


def lire_mode() -> str:
    return os.environ.get('MODE', 'prod')


def test_mode_dev(monkeypatch):
    monkeypatch.setenv('MODE', 'dev')
    assert lire_mode() == 'dev'


def test_mode_defaut():
    # monkeypatch non utilisé → env d'origine
    # assert lire_mode() == 'prod'  # dépend de l'env
    pass


---

## 7. `tmp_path` pour les fichiers

Fixture builtin qui fournit un `pathlib.Path` temporaire unique.

In [ ]:
from pathlib import Path


def ecrire_config(path: Path, data: str) -> None:
    path.write_text(data)


def test_ecrire(tmp_path):
    fichier = tmp_path / 'config.toml'
    ecrire_config(fichier, 'key = "value"')
    assert fichier.read_text() == 'key = "value"'


---

## 8. `caplog` pour les logs

Capture les messages de logging émis pendant le test.

In [ ]:
import logging

logger = logging.getLogger('mon_module')


def traiter() -> None:
    logger.warning('attention !')


def test_log_warning(caplog):
    with caplog.at_level(logging.WARNING, logger='mon_module'):
        traiter()
    assert 'attention' in caplog.text


---

## 9. Marks

Les marks sont des **étiquettes** sur les tests.

In [ ]:
@pytest.mark.slow
def test_gros_calcul():
    assert sum(range(10_000_000)) > 0


@pytest.mark.skip(reason='pas encore implémenté')
def test_futur():
    pass


@pytest.mark.xfail(reason='bug connu #123')
def test_bug():
    assert 1 + 1 == 3


Lancer uniquement les tests lents : `pytest -m slow`

Exclure les tests lents : `pytest -m 'not slow'`

---

## 10. Tester les exceptions

Utiliser `pytest.raises` comme context manager.

In [ ]:
def diviser(a: float, b: float) -> float:
    if b == 0:
        raise ValueError('division par zéro')
    return a / b


def test_diviser_par_zero():
    with pytest.raises(ValueError, match='division'):
        diviser(1, 0)


---

## Synthèse

| Outil | Rôle |
|---|---|
| `assert` | Vérification |
| `@pytest.fixture` | Contexte réutilisable |
| `@pytest.mark.parametrize` | Tests data-driven |
| `monkeypatch` | Modifier l'env pendant un test |
| `tmp_path` | Répertoire temporaire |
| `caplog` | Capturer les logs |
| `pytest.raises(E)` | Vérifier qu'une exception est levée |


### Règles à retenir

1. **Un test = un comportement.** Pas de test qui vérifie 10 choses.
2. **Fixtures pour le setup**, pas de `setUp/tearDown` de unittest.
3. **`parametrize` avant de dupliquer** un test.
4. **Tester les cas limites** : `0`, `None`, chaîne vide, liste vide.
5. **Les tests font partie du code** : qualité, nommage, couverture.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Test basique *(facile)*

Écrire `est_pair(n: int) -> bool` et 3 tests : `test_pair`, `test_impair`, `test_zero`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Pytest", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
def est_pair(n: int) -> bool:
    return n % 2 == 0

def test_pair():
    assert est_pair(4)

def test_impair():
    assert not est_pair(3)

def test_zero():
    assert est_pair(0)
```

</details>

### Exercice 2 — Parametrize *(moyen)*

Écrire `celsius_vers_fahrenheit(c: float) -> float` et un test paramétré avec au moins 4 cas.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Pytest", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import pytest

def celsius_vers_fahrenheit(c: float) -> float:
    return c * 9 / 5 + 32

@pytest.mark.parametrize('c, f', [(0, 32), (100, 212), (-40, -40), (37, 98.6)])
def test_conversion(c, f):
    assert pytest.approx(celsius_vers_fahrenheit(c), rel=1e-6) == f
```

</details>

### Exercice 3 — Fixture et exception *(moyen)*

Écrire une fixture `db_conn` qui retourne une connexion sqlite3 en mémoire. Écrire un test qui vérifie qu'une requête invalide lève `sqlite3.OperationalError`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Pytest", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import pytest, sqlite3

@pytest.fixture
def db_conn():
    conn = sqlite3.connect(':memory:')
    yield conn
    conn.close()

def test_requete_invalide(db_conn):
    with pytest.raises(sqlite3.OperationalError):
        db_conn.execute('SELECT * FROM table_inexistante')
```

</details>

### Exercice 4 — Test avec `tmp_path` et `monkeypatch` *(difficile)*

Écrire une fonction `charger_config(path: Path) -> dict` qui lit un TOML simple (utiliser `tomllib` stdlib 3.11+). Tester avec `tmp_path` en écrivant un fichier TOML dedans. Bonus : utiliser `monkeypatch` pour patcher `pathlib.Path.read_text`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Pytest", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import tomllib
from pathlib import Path
import pytest

def charger_config(path: Path) -> dict:
    return tomllib.loads(path.read_text())

def test_charger_config(tmp_path):
    f = tmp_path / 'config.toml'
    f.write_text('[database]\nhost = "localhost"\nport = 5432')
    config = charger_config(f)
    assert config['database']['host'] == 'localhost'
    assert config['database']['port'] == 5432
```

</details>

---

## Ressources externes

### Documentation officielle
- [pytest docs](https://docs.pytest.org/)
- [pytest fixtures](https://docs.pytest.org/en/stable/fixture.html)

### Lectures complémentaires
- *Python Testing with pytest* — Brian Okken (Pragmatic Bookshelf)